# Fine-tuning Laya on a grading job
A small open decision model, specialised on one grading job, against a general hosted one.
Every number below is computed in this run, except training itself, which takes about 85 minutes.

In [ ]:
%matplotlib inline
from nb_helpers import *
versions()

## The job, and the answer key
Each item is a technical question and an AI-written answer with one **planted** defect, or none (one in three is clean). The planted label is the answer key both engines are scored against.

In [ ]:
test = load_test()
test_summary(test)
show_item(test[7])

## Step 1. Combine everything that was generated
Haiku wrote every question and answer. The code-made rows plant surface defects with templates.

In [ ]:
from prepare_data import combine, drop_test_copies, weighted
SOURCES = ['data/train_judged.jsonl', 'data/train_dropped_near_test.jsonl', 'data/hybrid_haiku.jsonl', 'data/hybrid.jsonl']
sources_summary(SOURCES)
raw = combine(SOURCES)
print(len(raw), 'rows after removing duplicate ids')

## Step 2. Drop every near-copy of a test question
Haiku repeats itself inside a topic. Training on a reworded test question teaches the answer, not the task.

In [ ]:
kept, dropped = drop_test_copies(raw)
print(f'dropped {len(dropped)} of {len(raw)}, kept {len(kept)}')
show_near_copy(dropped)

## Step 3. Look at the class mix before training
**Lesson 1:** a clean-up step can break the mix. Clean questions are generic, so most near-copies were clean answers, and dropping them pushed the clean share well below the one in three the data was built with.

In [ ]:
from mix_report import plot, print_table, problems, shares
stages = [('as generated', raw, False), ('after dropping test copies', kept, False)]
print_table(stages)
plot(stages, 'plots/mix_step3.png');

## Step 4. Weight back to the design, and check every defect type
Weighting the 2,082 rows back to one clean in three puts them on design: that is the mix the checkpoint below was trained on. The target comes from how the data was generated, never from the test set's labels.

In [ ]:
design = weighted(kept, 'design')
stages = [('as generated', raw, False), ('after dropping test copies', kept, False),
          ('weighted: clean 1 in 3', design, True)]
print_table(stages)
print('; '.join(problems(shares(design, True))) or 'on design')
plot(stages, 'plots/mix_fixed.png', compare=(1, 2));

**Lesson 2:** check again after every change. Adding 40 natural examples to 11 defect types doubled their share; a model trained on that mix over-called those 11 types. Weighting clean vs defective does not catch it. Balancing every type does.

In [ ]:
natural, _ = drop_test_copies(combine(['data/natural_haiku.jsonl']))
more = kept + natural
stages = [('+ natural, unweighted', more, False),
          ('+ natural, weighted: clean 1 in 3', weighted(more, 'design'), True),
          ('+ natural, types balanced too', weighted(more, 'balanced'), True)]
print_table(stages)
for name, rows, w in stages:
    print(f'{name:36s}', '; '.join(problems(shares(rows, w))) or 'on design')
plot(stages, 'plots/mix_step4.png', compare=(1, 2));

## Step 5. Train
`train.py` repeats this check before its first step and refuses a skewed mix. The full run takes about 85 minutes on an M4 Pro; here are its first 25 steps, so you can watch it start. The checkpoint scored below is the finished run.

In [ ]:
import sys  # the kernel's own Python, so the shell command runs in this venv
!{sys.executable} train.py --train data/train_all.jsonl --mix design --max-steps 25 2>&1 | grep -v -i warn

## Step 6. Score on the 500 held-out items
Laya out of the box, then fine-tuned, on the same items and questions.

In [ ]:
from evaluate import evaluate
zero_shot = evaluate('base')

In [ ]:
tuned = evaluate('runs/all2082')

## Jev on the same 500 items, same short labels
A hosted API, so its latency includes the network.

In [ ]:
jev = await run_jev(test)

## The scoreboard

In [ ]:
scoreboard({'Laya, out of the box': zero_shot, 'Laya, fine-tuned': tuned, 'Jev': jev})

In [ ]:
print('pass/rework, agreement on the most confident share of items:')
for share, acc in tuned['verdict']['risk_coverage']:
    print(f'  top {share:>4.0%}: {acc:.3f}')

## Before you train on your own data
1. No test copies: drop every training item that is a near-copy of a test item.
2. Check the mix after **every** change to the data, clean vs defective and type by type, against how production looks, not against your test labels.
3. If you planted the labels, you already have them: a paid judge is a quality check, not a requirement.